# M34 — Build RAG

**Objective:** ground generation in retrieved evidence.

M33 already returns ranked evidence with identity. The useful whole
here is a **grounded answerer**:

`query → M33 retrieval → budgeted context pack → extractive synthesis
→ citations → support check or abstention`

A high cosine is a retrieval score. It is not labeled relevance and
it is not an answer. The provenance (index id, source hash, model/version)
must travel with every cited span. A fluent sentence is not grounded
unless a cited span supports the claim.

This notebook uses the bundled M33 exact index and a **local
extractive** synthesizer. Nothing is downloaded. No paid API is
required. Reranking, ANN infrastructure, and decoding labs stay
closed (M35, M36, M32).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a chunk id, an abstention, a failed
support check, or a named failure layer.

Do not download a generator, do not treat a cosine as an answer, and
do not open a required live model. If a failure can be diagnosed from
packed IDs versus gold support IDs, stay at that layer.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M34" / "rag_pipeline.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M33.semantic_search import (
    encode_query,
    load_canonical_corpus,
    load_canonical_index,
    search,
)
from missions.M34.rag_pipeline import (
    POLICY_NAIVE,
    answer_labeled,
    answer_query,
    classify_failure,
    evaluate_set,
    load_expected_payload,
    load_query_map,
    pack_context,
    repair_grounding,
    retrieve,
    synthesize,
    trace_report,
    verify_support,
)

corpus = load_canonical_corpus()
index = load_canonical_index()
QUERIES = load_query_map()
EXPECTED = load_expected_payload()

print("repository root:", ROOT)
print("index_id:", index.metadata.index_id)
print("model:", index.metadata.embedding.model)
print("version:", index.metadata.embedding.version)
print("metric:", index.metadata.embedding.metric)
print("normalization:", index.metadata.embedding.normalization)
print("source_hash:", index.metadata.source_hash[:16])
print("downloaded:", index.metadata.downloaded, "network:", index.metadata.network_required)
print("questions:", len(QUERIES))


## M33 / M31 boundary: evidence in, grounding here

M33 owns retrieval. M34 may call `search` and `RankedHit.as_evidence`.
It may not reimplement cosine or relabel a score as an answer.

M31 owns the training-versus-inference boundary. The extractive
synthesizer here is inference-time: `weights_updated=False`. How
tokens would be sampled from a live model is M32 and stays closed.

M35 will freeze these queries later. Do not rerank or retune chunk
size in this notebook.


## Frozen teaching fixtures

- Index: `v08-exact-memory` over `datasets/M33` (`m33.corpus.v1`)
- Encoder: `v06-teaching-meanpool` `v06.1`, cosine, L2
- Synthesizer: `extractive-span-v1`, policy `support_gated`
- Eval: `datasets/M34/questions.json` (`m34.eval.v1`)
- Holdout ids are not for policy tweaking

Support labels are independent of cosine. Do not edit them after
seeing ranks.


In [ ]:
for query_id in ("rag-reset-login", "rag-password-procedure", "rag-ticket-4412", "rag-ceo"):
    item = QUERIES[query_id]
    print(
        query_id,
        "split", item.split,
        "answerable", item.answerable,
        "support", item.support_chunk_ids,
    )
    print(" ", item.text)
holdout = [item.query_id for item in QUERIES.values() if item.split == "holdout"]
print("holdout ids", holdout)
print("eval_version", EXPECTED["eval_version"])


In [ ]:
labeled = QUERIES["rag-reset-login"]
encoded = encode_query(labeled.text, query_id=labeled.query_id)
reset_hits = search(
    index,
    encoded,
    top_k=3,
    live_corpus=corpus,
    query_id=labeled.query_id,
    enforce_freshness=True,
    enforce_provenance=True,
)
reset_evidence = tuple(hit.as_evidence() for hit in reset_hits.hits)
print("scored_candidates", reset_hits.scored_candidates)
for row in reset_evidence:
    print(
        row["rank"],
        round(row["score"], 4),
        row["chunk_id"],
        "index", row["index_id"],
        "hash", row["source_hash"][:12],
    )
    print(" ", row["text"])
reset_pack = pack_context(
    reset_evidence,
    query_id=labeled.query_id,
    query_text=labeled.text,
    retrieval_top_k=3,
    scored_candidates=reset_hits.scored_candidates,
)
print("--- packed context ---")
print(reset_pack.formatted)
print("pack ids", reset_pack.chunk_ids())
print("char_count", reset_pack.char_count())


### Inspect evidence before synthesis

The rows above are M33 evidence, not answers. Each row keeps
`document_id`, `chunk_id`, `span`, `index_id`, and `source_hash`.

If the later synthesizer cites a chunk, that citation must point at
one of these packed ids and the span must support the claim.


## Predict before running — first grounded answer

Timestamp a prediction before `run-rag`.

Query `rag-reset-login`: `How do I reset my login credentials?`

The index, retriever, and extractive policy are frozen. Predict:

- whether the system **answers** or **abstains**
- the citation `chunk_id` (not just "an account row")
- whether the cited text is the procedure span or a printer reset

Do not treat the top cosine as the answer. A guess from the packed
texts is the point.


In [ ]:
reset_trace = answer_labeled("rag-reset-login")
print(trace_report(reset_trace))
print("answer:", reset_trace.answer.text)
print("citations:", reset_trace.answer.citation_ids())
print("support_ok:", reset_trace.answer.support.ok)
print("eval_pass:", reset_trace.evaluation["eval_pass"])
print("weights_updated:", reset_trace.inference.weights_updated)
print("decoding:", reset_trace.inference.decoding)
assert reset_trace.answer.citations[0].index_id == reset_trace.index_id
assert reset_trace.answer.citations[0].source_hash == reset_trace.source_hash


### A grounded answer still has a retriever underneath

The citation must be a packed chunk id whose text supports the claim.
If you only remember the score, you are not doing RAG yet.


## Predict before running — with vs without retrieval

Timestamp a prediction before `run-with-without`.

**Change:** `retrieval_enabled=False` then `True` on the same
questions (`rag-legal-forbid`, `rag-fifty`, `rag-reset-login`).

**Invariant:** synthesizer policy and question texts stay fixed.

Predict, for the closed-book path:

- whether canonical synthesis answers or abstains
- whether any citation can be valid when the pack is empty


In [ ]:
pair_ids = ("rag-legal-forbid", "rag-fifty", "rag-reset-login")
with_rows = []
for query_id in pair_ids:
    off = answer_labeled(query_id, retrieval_enabled=False)
    on = answer_labeled(query_id, retrieval_enabled=True)
    with_rows.append((query_id, off, on))
    print(
        query_id,
        "off", off.answer.status, "cites", off.answer.citation_ids(),
        "on", on.answer.status, "cites", on.answer.citation_ids(),
        "pass", on.evaluation["eval_pass"],
    )
assert all(off.answer.abstained for _, off, _ in with_rows)
assert all(on.answer.answered and on.answer.support.ok for _, _, on in with_rows)


In [ ]:
labels = [query_id for query_id, _, _ in with_rows]
off_pass = [int(off.evaluation["eval_pass"]) for _, off, _ in with_rows]
on_pass = [int(on.evaluation["eval_pass"]) for _, _, on in with_rows]
off_support = [int(off.answer.support.ok and off.answer.answered) for _, off, _ in with_rows]
on_support = [int(on.answer.support.ok and on.answer.answered) for _, _, on in with_rows]
xpos = list(range(len(labels)))
width = 0.2
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.bar([x - 1.5 * width for x in xpos], off_pass, width, label="closed-book eval_pass")
ax.bar([x - 0.5 * width for x in xpos], on_pass, width, label="with-retrieval eval_pass")
ax.bar([x + 0.5 * width for x in xpos], off_support, width, label="closed-book supported answer")
ax.bar([x + 1.5 * width for x in xpos], on_support, width, label="with-retrieval supported answer")
ax.set_xticks(xpos)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylim(0, 1.15)
ax.set_ylabel("indicator (0/1)")
ax.set_title("Same questions, same policy: does retrieval change grounding?")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("closed-book statuses", [off.answer.status for _, off, _ in with_rows])
print("retrieval-on statuses", [on.answer.status for _, _, on in with_rows])


### Retrieval is an inference-time context change

Without a pack, the canonical extractive policy has nothing to copy.
That is abstention, not a clever parametric memory. Adding retrieval
does not update weights (M31). It changes the evidence window.


## Predict before running — top-k context

Timestamp a prediction before `run-topk`.

Query `rag-password-procedure`: `What should a user do if they forgot
their password?`

**Change:** `top_k` in `{1, 3, 5}`.

**Invariant:** index, encoder, and synthesizer policy stay fixed.

Predict:

- the top retrieved id at k=1
- whether gold support `doc-account-access::c1` is in the k=1 window
- whether k=3 can cite the procedure span even if it is not rank 1


In [ ]:
password_topk = []
for k in (1, 3, 5):
    result = answer_labeled("rag-password-procedure", top_k=k)
    password_topk.append(result)
    print(
        "k", k,
        "retrieved", result.retrieval_ids,
        "packed", result.pack.chunk_ids(),
        "status", result.answer.status,
        "cites", result.answer.citation_ids(),
        "layer", result.evaluation["primary"],
        "hit", result.evaluation["retrieval_hit"],
    )
password_k1 = password_topk[0]
password_k3 = password_topk[1]
assert password_k1.answer.abstained
assert password_k3.answer.citation_ids() == ("doc-account-access::c1",)


In [ ]:
ks = list(range(1, 6))
hits = []
packed = []
correct = []
for k in ks:
    result = answer_labeled("rag-password-procedure", top_k=k)
    hits.append(int(bool(result.evaluation["retrieval_hit"])))
    packed.append(int(bool(result.evaluation["packed_hit"])))
    correct.append(int(bool(result.evaluation["eval_pass"])))
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.plot(ks, hits, marker="o", label="gold support retrieved")
ax.plot(ks, packed, marker="s", label="gold support packed")
ax.plot(ks, correct, marker="^", label="eval_pass")
ax.set_xlabel("top-k")
ax.set_ylabel("indicator (0/1)")
ax.set_title("Password procedure: does a larger k add the supporting span?")
ax.set_ylim(-0.05, 1.15)
ax.set_xticks(ks)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print("retrieval_hit by k", list(zip(ks, hits)))
print("eval_pass by k", list(zip(ks, correct)))


### Rank-1 is allowed to be the wrong kind of neighbor

The problem statement can outscore the procedure. That is a retrieval
window fact, not proof that the procedure is absent from the corpus.
M35 may later ask whether a reranker would help. Here we only change
k.


## Predict before running — context budget

Timestamp a prediction before `run-budget`.

Keep `top_k=3` on `rag-password-procedure` so gold support is
retrieved. **Change:** `budget_chars=80`.

Packing stays in retrieval order and does not split chunks.

Predict:

- whether `retrieval_hit` stays true
- which chunk is dropped
- the primary failure layer


In [ ]:
budget80 = answer_labeled("rag-password-procedure", top_k=3, budget_chars=80)
print(trace_report(budget80))
print("retrieved", budget80.retrieval_ids)
print("packed", budget80.pack.chunk_ids(), "chars", budget80.pack.char_count())
print("dropped", budget80.pack.dropped_ids())
print("layer", budget80.evaluation["primary"], "retrieval_hit", budget80.evaluation["retrieval_hit"])
assert list(budget80.retrieval_ids) == list(password_k3.retrieval_ids)
assert budget80.evaluation["retrieval_hit"] is True
assert budget80.evaluation["packed_hit"] is False
assert budget80.evaluation["primary"] == "context"
assert list(budget80.pack.dropped_ids()) == ["doc-account-access::c1"]


### A retrieved span can still miss the pack

If you blame generation because the system abstained, you skipped a
layer. The gold id was retrieved, then dropped by the budget. That is
a context-assembly failure.


## Predict before running — unanswerable queries

Timestamp a prediction before `run-unanswerable`.

Queries: `rag-ceo` (`Who is the CEO of Valley Services?`) and
`rag-order-7` (order 7 refund amount). Corpus is fixed.

Predict:

- whether the top hit for `rag-ceo` is weather
- whether a high top score forces an answer
- what naive top-1 synthesis does with the same pack


In [ ]:
ceo_trace = answer_labeled("rag-ceo")
ceo_naive = answer_labeled("rag-ceo", policy=POLICY_NAIVE)
order_trace = answer_labeled("rag-order-7")
print("ceo gated", trace_report(ceo_trace))
print("ceo top score", round(ceo_trace.retrieval_scores[0], 4), ceo_trace.retrieval_ids[0])
print("ceo naive", ceo_naive.answer.status, ceo_naive.answer.text)
print("ceo naive layer", ceo_naive.evaluation["primary"])
print("order7", trace_report(order_trace))
assert ceo_trace.answer.abstained
assert ceo_trace.retrieval_ids[0] == "doc-weather::c0"
assert ceo_trace.retrieval_scores[0] > 0.99
assert ceo_naive.answer.answered
assert "Rain is expected" in ceo_naive.answer.text
assert order_trace.answer.abstained


### High retrieval score is not answer correctness

Weather is a neighbor of "Valley", not a CEO biography. The gated
policy abstains. Naive top-1 copies the distractor and looks fluent.
That is generation misuse, not a new fact about the world.


## Predict before running — retrieval miss

Timestamp a prediction before `run-miss`.

Query `rag-ticket-4412` at `top_k=1`. Gold support is
`doc-tickets::c0` (ticket 4412).

**Change:** the window is too small for the labeled span.
**Invariant:** gold support ids stay frozen.

Predict:

- the retrieved id at k=1
- whether gated synthesis answers or abstains
- what naive top-1 emits, and whether that failure is still
  retrieval-primary


In [ ]:
ticket_k1 = answer_labeled("rag-ticket-4412", top_k=1)
ticket_naive = answer_labeled("rag-ticket-4412", top_k=1, policy=POLICY_NAIVE)
ticket_k3 = answer_labeled("rag-ticket-4412", top_k=3)
print("k1 gated", trace_report(ticket_k1))
print("k1 naive", ticket_naive.answer.text, "layer", ticket_naive.evaluation["primary"])
print("k3 gated", trace_report(ticket_k3))
assert ticket_k1.retrieval_ids == ("doc-tickets::c1",)
assert ticket_k1.answer.abstained
assert ticket_k1.evaluation["primary"] == "retrieval"
assert ticket_naive.answer.text == "Ticket 4413 is waiting for inspection."
assert ticket_naive.evaluation["primary"] == "retrieval"
assert ticket_k3.answer.citation_ids() == ("doc-tickets::c0",)


### Classify retrieval before blaming generation

Naive top-1 will happily copy ticket 4413. The discriminating fact is
that 4412 never entered the window. Fixing the synthesizer cannot
create evidence that was not retrieved.


## Predict before running — citation support

Timestamp a prediction before `run-citation`.

Keep the `rag-reset-login` pack fixed. Compare a canonical answer
with `defect="unsupported_citation"`.

Predict:

- whether the defective **answer text** still looks like the procedure
- whether `verify_support` stays ok
- the primary failure layer


In [ ]:
citation_ok = reset_trace
citation_broken = answer_labeled("rag-reset-login", defect="unsupported_citation")
print("canonical cites", citation_ok.answer.citation_ids(), "ok", citation_ok.answer.support.ok)
print("broken text", citation_broken.answer.text)
print("broken cites", citation_broken.answer.citation_ids(), "ok", citation_broken.answer.support.ok)
print("broken layer", citation_broken.evaluation["primary"], "pass", citation_broken.evaluation["eval_pass"])
print("same pack", citation_ok.pack.chunk_ids() == citation_broken.pack.chunk_ids())
assert citation_ok.answer.support.ok
assert citation_broken.answer.text == citation_ok.answer.text
assert citation_broken.answer.support.ok is False
assert citation_broken.evaluation["primary"] == "citation"
assert list(citation_ok.pack.chunk_ids()) == list(citation_broken.pack.chunk_ids())


### Unsupported citations must fail even when the sentence is fluent

Evaluation is not a vibe check. If the cited span does not contain
the claim, the trace fails. That remains true when retrieval was
fine.


## Predict before running — held-out evaluation

Timestamp a prediction before `run-eval`.

`evaluate_set(split="holdout")` uses questions that were not used to
choose the extractive rules. Policy, index, and k stay at the
defaults.

Predict:

- whether unanswerable holdout items abstain
- whether invoice `99281` can still pass at default k=3
- whether this run is allowed to edit holdout labels


In [ ]:
holdout_report = evaluate_set(split="holdout")
print("n", holdout_report["n"], "pass", holdout_report["n_pass"], "rate", holdout_report["pass_rate"])
print("abstain", holdout_report["n_abstain"], "unsupported", holdout_report["n_unsupported_citation"])
print("held_out_untuned", holdout_report["held_out_untuned"])
for row in holdout_report["rows"]:
    print(
        row["query_id"],
        row["status"],
        "pass", row["eval_pass"],
        "cites", row["citation_ids"],
        "layer", row["primary"],
    )
assert holdout_report["n"] == 6
assert holdout_report["n_pass"] == 6
assert holdout_report["held_out_untuned"] is True


### Freeze the eval set before anyone tunes ranking

M35 is allowed to measure ranking quality on this set. It is not
allowed to relabel failures after seeing a new ranker. The baseline
you just ran is the handoff.


## Code reading — retrieve, pack, synthesize, cite, abstain, evaluate

Read `retrieve`, `pack_context`, `synthesize`, `verify_support`,
`classify_failure`, and `answer_query` in
`missions/M34/rag_pipeline.py`.

**Predict before running** the next cell:

1. whether packed items still carry `index_id` and `source_hash` from
   `as_evidence`
2. whether `pack_context` reorders hits to put gold support first
3. whether `verify_support` fails when the answer text is gold but
   the citation is a neighbor missing the claim stems

Do not search the file for a reranker, a chunk-overlap tuner, or a
sampling loop. Stay on evidence, budget, citations, and abstention.


In [ ]:
retrieve_src = inspect.getsource(retrieve)
pack_src = inspect.getsource(pack_context)
synth_src = inspect.getsource(synthesize)
verify_src = inspect.getsource(verify_support)
answer_src = inspect.getsource(answer_query)
print("retrieve calls search", "search(" in retrieve_src)
print("retrieve uses as_evidence", "as_evidence" in retrieve_src)
print("pack mentions reorder", "reorder" in pack_src.lower())
print("pack uses RankedHit.as_evidence path", "as_evidence" in pack_src)
print("synthesize has abstain", "abstain" in synth_src)
print("verify mentions unsupported_claim", "unsupported_claim" in verify_src)
print("answer_query records weights_updated", "weights_updated" in answer_src)
sample = reset_pack.items[0]
print("packed index_id", sample.index_id, "source_hash", sample.source_hash[:12])
print("budget drop is tail, not a new ranker", budget80.pack.dropped_ids())


## Predict before running — Controlled failure: unsupported citation

Timestamp a prediction before `run-failure`.

Canonical `rag-reset-login` already ran. **Change:** one named
defect, `defect="unsupported_citation"`. Query, index, retriever, and
pack stay fixed.

Predict:

- whether the emitted sentence still reads as a login reset
- whether support validation passes
- whether the primary layer is citation or retrieval


In [ ]:
defective = citation_broken
print(trace_report(defective))
print("text", defective.answer.text)
print("cites", defective.answer.citation_ids())
print("issues", [issue.kind for issue in defective.answer.support.issues])
print("retrieval_hit", defective.evaluation["retrieval_hit"], "packed_hit", defective.evaluation["packed_hit"])
assert defective.answer.answered
assert defective.answer.support.ok is False
assert defective.evaluation["primary"] == "citation"
assert defective.inference.defect == "unsupported_citation"
print("canonical still grounded", reset_trace.answer.support.ok)


### Diagnose before repair

Symptom: a fluent login-reset sentence came back with a citation.

Hypotheses worth separating: retrieval missed the procedure span; the
budget dropped it; the synthesizer invented a new claim; the citation
points at a packed neighbor that does not contain the claim stems.

The discriminating observation is the pack ids plus
`verify_support`. If gold support is packed and the cited neighbor
fails the stem check, the layer is citation.

Do not repair this by editing the query, relabeling support ids, or
opening M32/M35/M36.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Repair from the broken **answer** and the existing **pack**:
`repair_grounding(defective.answer, defective.pack)`.

Predict:

- whether the repaired citation is a packed span that supports the
  claim
- whether the original defective object stays unsupported
- what repair does if no packed span supports the claim (ticket
  invented-support at k=1)


In [ ]:
repaired_answer = repair_grounding(defective.answer, defective.pack)
print("repaired status", repaired_answer.status)
print("repaired cites", repaired_answer.citation_ids(), "ok", repaired_answer.support.ok)
print("repaired text", repaired_answer.text)
still_broken = verify_support(defective.answer, defective.pack)
print("original still broken", still_broken.ok is False)

invented = answer_labeled("rag-ticket-4412", top_k=1, defect="invented_support")
invented_layers = classify_failure(invented, QUERIES["rag-ticket-4412"])
invented_repaired = repair_grounding(invented.answer, invented.pack)
print("invented layers", invented_layers["layers"], "primary", invented_layers["primary"])
print("invented repair", invented_repaired.status, invented_repaired.abstain_reason)
assert repaired_answer.citation_ids() == ("doc-account-access::c1",)
assert repaired_answer.support.ok
assert still_broken.ok is False
assert invented_repaired.abstained


### Rebind or abstain; do not redecorate

The smallest repair uses the broken objects. If a packed span
supports the claim, recast the citation. If none does, abstain.
Do not add a reranker to hide an unsupported citation.


## Evidence contract

Your evidence log must include timestamped predictions, a grounded
trace with source IDs, the with/without pair, top-k and budget
layers, unanswerable abstention, a retrieval miss, a failed
unsupported citation, the repair, and the held-out eval summary.

Fixture numbers in `expected.json` are not learner evidence.


## No-AI gate

Close this notebook and complete `missions/M34/no_ai_gate.md` from a
blank page without AI-generated code, calculations, prose, or
diagrams.

Use only `datasets/M34/transfer.json`. Classify four traces, assemble
a pack by hand, write a supported answer, reject a fluent unsupported
claim, and state why RAG does not guarantee truth.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Author the V09 grounding contract in `missions/M34/adr_prompt.md`
using `templates/ADR.md`. Compare fail-closed extractive grounding,
fluent top-1 without support checks, and a required paid live model.

**Status:** [UNFILLED BY LEARNER]

Formal engineering review uses `missions/M34/review_brief.md`. This
notebook is not a review signature.


## M33 → M34 handoff

M35 receives:

- frozen questions and support/relevance labels in `datasets/M34`
- traces with retrieval ids, packed ids, citations, and layer names
- a baseline that already fails unsupported citations

M35 may change ranking and chunking. It may not silently relabel
these questions to improve an average. M36 still does not belong in
this notebook.


## Mission summary prompt

In your own words: what does it mean for an answer to be grounded?
Where did high retrieval score disagree with correctness? Which
failure was retrieval, which was context, which was citation? What
would you refuse to ship?

Leave the answers in your evidence log, not in this repository.


In [ ]:
assert reset_hits.ids()[0] == "doc-account-access::c1"
assert reset_evidence[0]["index_id"] == index.metadata.index_id
assert reset_trace.answer.citation_ids() == ("doc-account-access::c1",)
assert all(off.answer.abstained for _, off, _ in with_rows)
assert password_k1.evaluation["primary"] == "retrieval"
assert password_k3.answer.citation_ids() == ("doc-account-access::c1",)
assert budget80.evaluation["primary"] == "context"
assert ceo_trace.answer.abstained and ceo_trace.retrieval_scores[0] > 0.99
assert "Rain is expected" in ceo_naive.answer.text
assert ticket_k1.evaluation["primary"] == "retrieval"
assert ticket_naive.answer.text.startswith("Ticket 4413")
assert citation_broken.evaluation["primary"] == "citation"
assert repaired_answer.support.ok
assert holdout_report["n_pass"] == 6
print("M34 integrity checks passed")
